In [ ]:
!pip install openai==0.28 --force-reinstall --quiet
!pip install openai scikit-learn --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.3/147.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.7/223.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.5

In [3]:
# 라이브러리 설치
!pip install openai==0.28 firebase-admin scikit-learn --quiet

# 라이브러리 임포트
import firebase_admin
from firebase_admin import credentials, db
import openai
import requests
import re
import getpass
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# 기본 설정
FIREBASE_URL = "https://apptrackerdemo-569ea-default-rtdb.firebaseio.com"
cred_path = getpass.getpass("Enter path to Firebase admin SDK JSON:")
openai.api_key = getpass.getpass("Enter your OpenAI API Key: ")
TODAY = "2025-6-6"

# Firebase 초기화
if not firebase_admin._apps:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_URL})

ref = db.reference()
CATEGORY_SCORES = {
    "공부": 100, "정보수집": 30, "생산": 50,
    "SNS": 20, "엔터테인먼트": 5, "기타": 0,
}

# GPT 요약 요청
def ask_gpt_summary(messages):
    prompt = (
        "다음은 사용자의 앱 활동 로그입니다:\n"
        + "\n".join(messages) +
        "\n\n이 활동을 각 로그 별로 한 문장으로 요약하고, "
        "활동 종류를 무조건 [공부], [엔터테인먼트], [SNS], [정보수집], [생산], [기타] 중 하나로만 표시해 주세요.\n"
        "형식: [카테고리] 요약문"
    )
    try:
        res = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}]
        )
        return res["choices"][0]["message"]["content"].strip().split('\n')
    except Exception as e:
        print("GPT 요청 실패:", e)
        return ["[기타] 처리 실패"] * len(messages)

# 카테고리 및 시간 추출
def extract_category(summary):
    match = re.search(r"\[(.*?)\]", summary)
    return match.group(1).strip() if match else "기타"

def extract_minutes(text):
    h = re.search(r"(\d+(\.\d+)?)시간", text)
    m = re.search(r"(\d+(\.\d+)?)분", text)
    hours = float(h.group(1)) if h else 0
    minutes = float(m.group(1)) if m else 0
    return int(hours * 60 + minutes)  # 소수점 버림

# 클러스터링 함수
def cluster_logs(logs):
    messages = [log["message"] for log in logs.values() if "message" in log]
    if len(messages) < 2:
        return {0: messages}
    tfidf = TfidfVectorizer()
    X = tfidf.fit_transform(messages)
    n_clusters = min(len(messages), 6)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
    kmeans.fit(X)
    clusters = {i: [] for i in range(n_clusters)}
    for idx, label in enumerate(kmeans.labels_):
        clusters[label].append(messages[idx])
    return clusters

# usageStats → logs 변환
def convert_usageStats_to_logs(date_str):
    usage_ref = ref.child(f'usageStats/{date_str}')
    logs_ref = ref.child("logs")
    usage_data = usage_ref.get()
    if not usage_data:
        print(f"usageStats/{date_str} 에 데이터 없음")
        return
    for uid, apps in usage_data.items():
        for package_name, data in apps.items():
            used_time_min = round(data.get("usedTimeMillis", 0) / 1000 / 60, 1)
            log_msg = f"{package_name} 앱을 {used_time_min}분 동안 사용했습니다."
            log_id = f"log_{package_name.replace('.', '_')}"
            logs_ref.child(uid).child(date_str).child(log_id).set({"message": log_msg})
        print(f"usageStats → logs 변환 완료: {uid} ({date_str})")

# 분석 및 포인트 적립 함수
def analyze_logs_and_update_points():
    logs_ref = ref.child("logs")
    users = logs_ref.get()
    print(f"\nSTEP 2: 활동 로그 기반 포인트 분석 (클러스터링 포함)\n")
    for uid, dates in users.items():
        total_score = 0
        for date, log_dict in dates.items():
            if date != TODAY:
                continue
            clusters = cluster_logs(log_dict)
            log_map = {log["message"]: log_id for log_id, log in log_dict.items() if "message" in log}
            for cluster_msgs in clusters.values():
                gpt_responses = ask_gpt_summary(cluster_msgs)
                for i, summary_line in enumerate(gpt_responses):
                    category = extract_category(summary_line)
                    minutes = extract_minutes(summary_line)
                    multiplier = 0 if minutes == 0 else max(1, minutes // 60)
                    score = CATEGORY_SCORES.get(category, 0) * multiplier
                    total_score += score
                    print(f" - {summary_line} → [{category}] / {minutes}분 / +{score}점")
                    if i < len(cluster_msgs):
                        msg = cluster_msgs[i]
                        log_id = log_map.get(msg)
                        if log_id:
                            logs_ref.child(uid).child(date).child(log_id).update({"category": category})
        ref.child("users").child(uid).child("points").set(total_score)
        print(f"\n✅ {uid}에게 {total_score}점 적립 완료\n")

    print("포인트 업데이트 완료\n\n실시간 사용자 랭킹:")
    res = requests.get(f"{FIREBASE_URL}/users.json")
    scores = {uid: val.get("points", 0) for uid, val in res.json().items()}
    for i, (uid, score) in enumerate(sorted(scores.items(), key=lambda x: x[1], reverse=True), 1):
        print(f"{i}위 - {uid} : {score}점")

#테스트용 로그 업로드
def upload_test_logs():
    logs = {
        "user123": {
            "log1": "PUBG 게임을 2시간 플레이했습니다",
            "log2": "2시간 동안 수학 문제를 풀었습니다",
            "log3": "1시간 동안 한글 파일을 사용하여 문서 제작"
        },
        "user456": {
            "log1": "과학 프로젝트를 공부했습니다",
            "log2": "3시간 동안 오버워치를 했습니다",
            "log3": "2시간 30분 동안 인스타그램을 사용했습니다",
            "log4": "1시간 동안 구글을 사용하여 정보를 수집"
        }
    }
    for uid, user_logs in logs.items():
        for log_id, message in user_logs.items():
            ref.child("logs").child(uid).child(TODAY).child(log_id).set({"message": message})
            print(f"  저장됨: {uid}/{TODAY}/{log_id} → {message}")
    print(" 테스트 로그 업로드 완료")

#전체 실행 흐름
convert_usageStats_to_logs(TODAY)
upload_test_logs()
analyze_logs_and_update_points()


usageStats → logs 변환 완료: 0252aecd22e4cd2f (2025-6-6)
  ✅ 저장됨: user123/2025-6-6/log1 → PUBG 게임을 2시간 플레이했습니다
  ✅ 저장됨: user123/2025-6-6/log2 → 2시간 동안 수학 문제를 풀었습니다
  ✅ 저장됨: user123/2025-6-6/log3 → 1시간 동안 한글 파일을 사용하여 문서 제작
  ✅ 저장됨: user456/2025-6-6/log1 → 과학 프로젝트를 공부했습니다
  ✅ 저장됨: user456/2025-6-6/log2 → 3시간 동안 오버워치를 했습니다
  ✅ 저장됨: user456/2025-6-6/log3 → 2시간 30분 동안 인스타그램을 사용했습니다
  ✅ 저장됨: user456/2025-6-6/log4 → 1시간 동안 구글을 사용하여 정보를 수집
📌 테스트 로그 업로드 완료

STEP 2: 활동 로그 기반 포인트 분석 (클러스터링 포함)

 - [SNS] 사용자가 com_google_android_apps_messaging 앱을 0.0분 동안 사용하였습니다. → [SNS] / 0분 / +0점
 - [엔터테인먼트] 사용자가 YouTube 앱을 1.9분 동안 사용하였습니다. → [엔터테인먼트] / 1분 / +5점
 - [정보수집] 사용자가 com_android_chrome 앱을 1.8분 동안 사용하였습니다. → [정보수집] / 1분 / +30점
 - [정보수집] 사용자가 com_google_android_gm 앱을 0.6분 동안 사용했습니다. → [정보수집] / 0분 / +0점
 - [정보수집] 사용자가 Google Photos 앱을 0.1분 동안 사용하여 사진을 봤습니다. → [정보수집] / 0분 / +0점
 - [기타] com_example_apptracker 앱을 사용한 시간은 0.0분입니다. → [기타] / 0분 / +0점
 - [정보수집] com_google_android_apps_nexuslauncher 앱을 사용한 시간은 1.4분입니다